In [1]:
import pandas as pd
import numpy as np
import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

print("ML libraries imported successfully!")

ML libraries imported successfully!


In [3]:
df = pd.read_csv(
    "../data/processed/churn_feature_engineered.csv"
)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (7043, 30)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Churn,TenureGroup,AvgMonthlySpend,ServiceCount,SecuritySupportCount,IsNewCustomer,HighMonthlyCharge,IsMonthToMonth,IsElectronicCheck,HighRiskCustomer
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,0-12 Months,29.850000,1,1,1,0,1,1,1
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,25-48 Months,55.573529,3,2,0,0,0,0,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Yes,0-12 Months,54.075000,3,2,1,0,1,0,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,25-48 Months,40.905556,3,3,0,0,0,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Yes,0-12 Months,75.825000,1,0,1,1,1,1,1


In [4]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 30 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customerID            7043 non-null   str    
 1   gender                7043 non-null   str    
 2   SeniorCitizen         7043 non-null   int64  
 3   Partner               7043 non-null   str    
 4   Dependents            7043 non-null   str    
 5   tenure                7043 non-null   int64  
 6   PhoneService          7043 non-null   str    
 7   MultipleLines         7043 non-null   str    
 8   InternetService       7043 non-null   str    
 9   OnlineSecurity        7043 non-null   str    
 10  OnlineBackup          7043 non-null   str    
 11  DeviceProtection      7043 non-null   str    
 12  TechSupport           7043 non-null   str    
 13  StreamingTV           7043 non-null   str    
 14  StreamingMovies       7043 non-null   str    
 15  Contract              7043 non-n

In [5]:
print("Missing values:", df.isnull().sum().sum())

Missing values: 0


In [6]:
if "customerID" in df.columns:
    df = df.drop("customerID", axis=1)

print("Shape:", df.shape)

Shape: (7043, 29)


In [7]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print(df["Churn"].value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [8]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 28)
y shape: (7043,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 28)
X_test : (1409, 28)
y_train: (5634,)
y_test : (1409,)


In [10]:
categorical_cols = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical Features:")
print(categorical_cols)

print("\nNumerical Features:")
print(numerical_cols)

Categorical Features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup']

Numerical Features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'ServiceCount', 'SecuritySupportCount', 'IsNewCustomer', 'HighMonthlyCharge', 'IsMonthToMonth', 'IsElectronicCheck', 'HighRiskCustomer']


In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols
        )
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [12]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed X_train:", X_train_processed.shape)
print("Processed X_test :", X_test_processed.shape)

Processed X_train: (5634, 57)
Processed X_test : (1409, 57)


In [13]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_processed,
    y_train
)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [14]:
logistic_pred = logistic_model.predict(
    X_test_processed
)

logistic_prob = logistic_model.predict_proba(
    X_test_processed
)[:, 1]

In [15]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(
    X_train_processed,
    y_train
)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [16]:
rf_pred = rf_model.predict(
    X_test_processed
)

rf_prob = rf_model.predict_proba(
    X_test_processed
)[:, 1]

In [17]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_processed,
    y_train
)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [18]:
xgb_pred = xgb_model.predict(
    X_test_processed
)

xgb_prob = xgb_model.predict_proba(
    X_test_processed
)[:, 1]

In [19]:
print("Dataset shape:", df.shape)

print("Duplicate rows:", df.duplicated().sum())

print("\nChurn distribution:")
print(df["Churn"].value_counts())

Dataset shape: (7043, 29)
Duplicate rows: 22

Churn distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [20]:
print("Train + Test:",
      len(X_train) + len(X_test))

print("Original:",
      len(df))

Train + Test: 7043
Original: 7043


In [21]:
import joblib

joblib.dump(
    logistic_model,
    "../models/logistic_model.pkl"
)

joblib.dump(
    rf_model,
    "../models/random_forest_model.pkl"
)

joblib.dump(
    xgb_model,
    "../models/xgboost_model.pkl"
)

joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

print("All models and preprocessor saved successfully!")

All models and preprocessor saved successfully!


In [22]:
feature_names = preprocessor.get_feature_names_out()

joblib.dump(
    feature_names.tolist(),
    "../models/feature_columns.pkl"
)

print("Feature columns saved successfully!")

Feature columns saved successfully!
